This particular code is installing the Spacy Library which is an open source NLP tool kit this is use usually for tokenization, part of speech tagging, and name entity recognition or NER but for our activity this spacy helps us to focus on NER since this provides us an NER engine that can detect entities such as people, organizations, locations, dates, and also the percentages we have in our sample text.

In [ ]:
!pip install spacy

And for this part, the code downloads an English-language model for spaCy named en_core_web_sm. Whereas spaCy is the framework, it still requires a trained model in order to analyze text and perform entity recognition. This model contains patterns and rules for the English language that allow spaCy to work with Tokenization, POS tagging, and above all Named Entity Recognition (NER). This foremost consideration for our exercise can be viewed as a prerequisite for modeling the entities detected by SpaCy, like persons, organizations, locations, dates, percentages, etc., with our sample text. That model is a precondition for spaCy modeling any English text for NER at all.

In [ ]:
!python3 -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


And for the next part of code, here is where we bring in all the important tools we need in our work. First, we bring in spaCy, which is the main NLP library we are using we also import displacy from spaCy, which is an inbuilt visualization tool that can render named entities in a text more visually. Second, we import different evaluation criteria from sklearn.metrics, among them accuracy, precision, recall, and F1-score. These metrics will later serve us in measuring the Named Entity Recognition (NER) model performance by comparing its prediction with our own manually annotated ground truth. Lastly, we load the English model en_core_web_sm into a variable called nlp. This is where we start giving spaCy the real knowledge of the English language, thereby performing text processing and entity recognition of different types in our dataset like PERSON, ORG, DATE, LOC, etc.

In [ ]:
import spacy
from spacy import displacy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nlp = spacy.load('en_core_web_sm')

This code checks, whether NER component is included in the loaded spaCy model. If so, it fetches all the available entity labels like PERSON, ORG, GPE, DATE, etc., and prints them all out for us to know what categories the model can recognize. If not, it prints a warning message. This actually lets us confirm what kind of entities the model can detect before we run any tests for the data set.

In [ ]:
if "ner" in nlp.pipe_names:
    ner_labels = nlp.get_pipe("ner").labels
    print("Available Entity Categories")
    print(", ".join(sorted(ner_labels)))
else:
    print("NER pipeline component not found in the loaded model.")

Available Entity Categories
CARDINAL, DATE, EVENT, FAC, GPE, LANGUAGE, LAW, LOC, MONEY, NORP, ORDINAL, ORG, PERCENT, PERSON, PRODUCT, QUANTITY, TIME, WORK_OF_ART


And for this particular code it creates a list dubbed text_set which contains not less than three excerpts from long pieces of writing. These pieces are formed along the lines of long-winded financial and corporate documents such as earning announcements and statements from chief executive officers. They are important in our work as they will contain a lot many named entities that we want to capture with ner. For instance, some people such as Keith Waddell and Tim Cook; organizations such as Robert Half and Apple; locations such as the United States and Greater China; dates such as July 1, 2023; and yet other numbers like percentages, revenues, and stock prices. By putting these pieces together, we provide our model with realistic data to verify how well it will perform under such conditions of locating and classifying different types of entities in a domain-specific context.

In [ ]:
text_set = [
  "Keith Waddell, President and Chief Executive Officer of Robert Half, opened the call by thanking Mike Buckley, CFO. Waddell noted that U.S elections trade policy uncertainty had led economists to cut their growth forecasts for 2025. In March and April, the company reduced staffing levels at corporate offices across the United States. These changes created annual cost savings of $80 million. He emphasized that Protiviti, a subsidiary, achieved year-over-year revenue growth for the third quarter, despite market headwinds. Business confidence, which had surged after the U.S. elections, has now moderated.",

  "WhiteHorse Finance reported Q2 2025 results that fell short of analyst expectations. The company announced an earnings per share of $0.10, compared to the forecasted $0.3079, a 67.52% negative surprise. Revenue was $18.84 million, missing the forecast of $19.44 million by 3.09%. Following the announcement, the company’s stock price closed at $8.52, down 2.96%. According to InvestingPro, four analysts revised their estimates downward, highlighting continued challenges in the United States markets.",

  "Apple today announced financial results for its fiscal 2023 third quarter ended July 1, 2023. The Company posted quarterly revenue of $81.8 billion, down 1 percent year over year, and quarterly earnings per diluted share of $1.26. “We are happy to report that we had an all-time revenue record in Services during the quarter, driven by over 1 billion paid subscriptions,” said Tim Cook, Apple’s CEO. “We continue to invest in innovation and long-term growth, particularly in areas such as AI and silicon design.” CFO Luca Maestri added: “Our year-over-year business performance improved compared to the March quarter, and we generated nearly $26 billion in operating cash flow. We returned over $24 billion to shareholders, while continuing to invest in our growth plans.” Apple noted strong sales in North America, Europe, and Greater China, with emerging markets like India showing record performance."
]


to explain this code it contains the ground_truth which serves as our reference for manually annotating the texts we used earlier in text_set. We mention a list of entities for every text which includes its defined labels like PERSON, ORG, LOC, DATE, MONEY, PRODUCT, PERCENT, and EVENT. These annotations are important because they serve as the benchmark against which we will evaluate the spaCy model’s predictions. In other words, this is the “answer key” that tells us what entities truly exist in each passage. By doing this, we make it possible to compare the model’s output with the actual expected entities. This step, therefore, will be necessary in calculating accuracy, precision, recall, and F1-score later on.

In [ ]:
ground_truth = {
    text_set[0]: [
        ("Keith Waddell", "PERSON"),
        ("Mike Buckley", "PERSON"),
        ("Robert Half", "ORG"),
        ("U.S.", "LOC"),
        ("United States", "LOC"),
        ("2025", "DATE"),
        ("March", "DATE"),
        ("April", "DATE"),
        ("Protiviti", "ORG"),
        ("third quarter", "DATE"),
        ("U.S elections", "EVENT"),
        ("$80 million", "MONEY")
    ],

    text_set[1]: [
        ("WhiteHorse Finance", "ORG"),
        ("Q2 2025", "DATE"),
        ("$0.10", "MONEY"),
        ("$0.3079", "MONEY"),
        ("67.52%", "PERCENT"),
        ("$18.84 million", "MONEY"),
        ("$19.44 million", "MONEY"),
        ("3.09%", "PERCENT"),
        ("$8.52", "MONEY"),
        ("2.96%", "PERCENT"),
        ("InvestingPro", "ORG"),
        ("United States", "LOC")
    ],

    text_set[2]: [
        ("Apple", "ORG"),
        ("2023", "DATE"),
        ("third quarter", "DATE"),
        ("July 1, 2023", "DATE"),
        ("$81.8 billion", "MONEY"),
        ("1 percent", "PERCENT"),
        ("$1.26", "MONEY"),
        ("Services", "PRODUCT"),
        ("1 billion", "MONEY"),
        ("Tim Cook", "PERSON"),
        ("AI", "PRODUCT"),
        ("Luca Maestri", "PERSON"),
        ("March quarter", "DATE"),
        ("$26 billion", "MONEY"),
        ("$24 billion", "MONEY"),
        ("North America", "LOC"),
        ("Europe", "LOC"),
        ("Greater China", "LOC"),
        ("India", "LOC")
    ]
}


This part is where we prepare two empty lists: true_labels and pred_labels. The idea is simple — true_labels will store the correct entity labels from our manually annotated ground truth, while pred_labels will store the labels predicted by the spaCy model. Later on, we’ll compare these two lists side by side to evaluate how well the model performs.

In [ ]:
true_labels = []
pred_labels = []

This code basically shows us how spaCy predicts entities in our given texts and then lines them up against our ground truth labels so we can later evaluate how well our model is doing. For each text, the code prints out the entities spaCy finds, such as names, dates, money amounts, or organizations, along with their labels and explanations. It also highlights them visually with displacy. After that, it carefully searches our manually annotated ground truth labels in the text, matches them to the correct spans, and assigns the proper labels per token while marking all non-entities as "O". At the same time, it collects spaCy’s predictions per token. Both lists (true_labels and pred_labels) are then extended for later scoring. For example, in the sample output, the model detected “Keith Waddell” as a PERSON and “$80 million” as MONEY. But it also made mistakes like tagging “AI” as a GPE instead of recognizing it as a PRODUCT, And Robert Half as an Person instead of an ORG This shows us that the code not only extracts predictions but also sets us up to measure accuracy by comparing these results with the correct annotations we already prepared.

In [ ]:
for text in text_set:
    doc = nlp(text)

    if doc.ents:
        for ent in doc.ents:
            print(f"Predicted -> Text: '{ent.text}' | Label: {ent.label_} ({spacy.explain(ent.label_)})")
        displacy.render(doc, style="ent", jupyter=True)
    else:
        print("No named entities detected in the text.")

    # Convert gold entities into spans for alignment
    gold_spans = []
    for (ent_text, ent_label) in ground_truth[text]:
        start = text.find(ent_text)
        if start != -1:
            end = start + len(ent_text)
            gold_spans.append((start, end, ent_label))

    # Assign gold labels per token
    token_truth = []
    for token in doc:
        assigned_label = "O"
        for (start, end, ent_label) in gold_spans:
            # Check if the token is within the span of a ground truth entity
            if token.idx >= start and token.idx + len(token.text) <= end:
                assigned_label = ent_label
                break
        token_truth.append(assigned_label)


    token_preds = []
    for token in doc:
        if token.ent_type_:
            token_preds.append(token.ent_type_)
        else:
            token_preds.append("O")

    true_labels.extend(token_truth)
    pred_labels.extend(token_preds)

Predicted -> Text: 'Keith Waddell' | Label: PERSON (People, including fictional)
Predicted -> Text: 'Robert Half' | Label: PERSON (People, including fictional)
Predicted -> Text: 'Mike Buckley' | Label: PERSON (People, including fictional)
Predicted -> Text: 'CFO' | Label: ORG (Companies, agencies, institutions, etc.)
Predicted -> Text: 'Waddell' | Label: PERSON (People, including fictional)
Predicted -> Text: 'U.S.' | Label: GPE (Countries, cities, states)
Predicted -> Text: '2025' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'March' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'April' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'the United States' | Label: GPE (Countries, cities, states)
Predicted -> Text: 'annual' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: '$80 million' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: 'Protiviti' | Label: ORG (Compani

Predicted -> Text: 'Q2 2025' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: '0.10' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: '0.3079' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: '67.52%' | Label: PERCENT (Percentage, including "%")
Predicted -> Text: '$18.84 million' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: '$19.44 million' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: '3.09%' | Label: PERCENT (Percentage, including "%")
Predicted -> Text: '8.52' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: '2.96%' | Label: PERCENT (Percentage, including "%")
Predicted -> Text: 'InvestingPro' | Label: ORG (Companies, agencies, institutions, etc.)
Predicted -> Text: 'four' | Label: CARDINAL (Numerals that do not fall under another type)
Predicted -> Text: 'the United States' | Label: GPE (Countries, cities, states)


Predicted -> Text: 'Apple' | Label: ORG (Companies, agencies, institutions, etc.)
Predicted -> Text: 'today' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'fiscal 2023 third quarter ended July 1, 2023' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'Company' | Label: ORG (Companies, agencies, institutions, etc.)
Predicted -> Text: 'quarterly' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: '$81.8 billion' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: '1 percent' | Label: PERCENT (Percentage, including "%")
Predicted -> Text: 'year' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'quarterly' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: '1.26' | Label: MONEY (Monetary values, including unit)
Predicted -> Text: 'the quarter' | Label: DATE (Absolute or relative dates or periods)
Predicted -> Text: 'over 1 billion' | Label: MONEY (Monetary 

And lastly to verify on how our model performs this code calculates it and displays the evaluation metrics to see how well the spaCy model performed on our texts. Using the lists true_labels and pred_labels that we built earlier, it computes accuracy, which tells us the overall percentage of tokens labeled correctly. Then it calculates precision, which shows how many of the predicted entities were actually correct, and recall, which measures how many of the real entities the model successfully found. Finally, it computes the F1 score, which balances precision and recall into a single score to give a fair view of overall performance.

In [ ]:
# --- Evaluation Results ---
print("\n--- Evaluation Metrics ---")
print("Accuracy:", accuracy_score(true_labels, pred_labels))
print("Precision:", precision_score(true_labels, pred_labels, average="weighted", zero_division=0))
print("Recall:", recall_score(true_labels, pred_labels, average="weighted", zero_division=0))
print("F1 Score:", f1_score(true_labels, pred_labels, average="weighted", zero_division=0))


--- Evaluation Metrics ---
Accuracy: 0.8955613577023499
Precision: 0.932865848009451
Recall: 0.8955613577023499
F1 Score: 0.9077571069438565
